In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from utils.cancel_func import get_cross_section_sh, first_lock_sh, first_unlock_sh, first_lock_sh_id, time_add, time_sub

predict_t
t 检查自己单子的交易所时间戳
lock_time
lock_id
limit_up

最新一条order/trade 

order/trade ntime 
order/trade nprice
order/trade nvolume
order norderid
trade nbidorderid
trade naskorderid
order/trade nindex

order/trade q_bsflag
order q_ordertype
order/trade q_deltatime
order/trade q_lasttime

order q_maxordervol
order q_maxwithdrawvol
trade q_maxtradevol
order q_nordercnt
trade q_ntradecnt

In [ ]:
递推：

当前封板量 c_fbqty : 当前封板买一的量
当前总涨停挂单量（含撤） c_ztbuyqty : 封板时刻及以后，到当前涨停价的总买入委托单量，不减去撤单 
当前总涨停成交量 c_zttradeqty : 封板时刻及以后，到当前涨停价的总成交量
当前总涨停撤单量 c_ztcancelqty : 封板时刻及以后，到当前涨停价的总买入委托撤单量


In [ ]:
# 把毫秒转换为原始时间
def convert_to_time(milliseconds: int):
    total_seconds = milliseconds // 1000
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60
    millis = milliseconds % 1000
    
    original_format = (hours * 10000000 + 
                      minutes * 100000 + 
                      seconds * 1000 + 
                      millis)
    
    return original_format

# 把time转换成毫秒
def convert_to_ms(time_value: int):
    hours = time_value // 10000000
    minutes = (time_value // 100000) % 100
    seconds = (time_value // 1000) % 100
    milliseconds = time_value % 1000

    total_milliseconds = (hours * 3600 + minutes * 60 + seconds) * 1000 + milliseconds
    return total_milliseconds

# 60进制下int类型的时间减法(转毫秒)
def time_sub(a: int, b: int):
    if a >= 130000000 and b <= 113000000:
        return convert_to_ms(113000000) - convert_to_ms(b) + convert_to_ms(a) - convert_to_ms(130000000)
    else:
        return convert_to_ms(a) - convert_to_ms(b)

# 60进制下int类型的时间加法
def time_add(a: int, b: int):
    result = convert_to_time(convert_to_ms(a) + convert_to_ms(b))
    # 如果起始在 11:30 前，直接相加的结果在 11:30 到 13:00 之间
    if a < 113000000 and result > 113000000 and result < 130000000:
        result = time_add(130000000, convert_to_time(time_sub(result, 113000000)))
    # 如果起始在 11:30 前，直接相加的结果在 13:00 之后
    elif a < 130000000 and result > 130000000:
        result = time_add(130000000, result)
    if result > 150000000:
        return 150500000
    return result

In [ ]:
class FactorCalculator():
    def __init__(self, order, trade, predict_t, t):
        self.order = order
        self.trade = trade
        self.predict_t = predict_t # 预测应该撤单的时刻
        self.t = t # 自己的单子到达订单簿的时间(t70)
        self.lock_time = first_lock_sh(order, trade)
        self.lock_id = first_lock_sh_id(order, trade)
        self.limit_up = order[order['bizIndex'] == self.lock_id]['orderPrice'].iloc[0].item()
        self.cross_section = get_cross_section_sh(self.order, self.trade)

    # 封板时刻的封单额
    def factor001(self):
        return self.cross_section[self.cross_section['time'] == self.lock_time]['委买总额'].iloc[-1]

    # predict_t时刻的封单额
    def factor002(self):
        return self.cross_section[self.cross_section['time'] == self.predict_t]['委买总额'].iloc[-1]
    
    # predict_t时刻之间的最大封单额
    def factor003(self):
        return self.cross_section[self.cross_section['time'] <= self.predict_t]['委买总额'].max()
    
    # predict_t时刻订单簿的成交情况
    def factor004(self):
        order_ids = self.order[(self.order['time'] >= self.lock_time) &
                    (self.order['time'] <= self.predict_t) &
                    (self.order['orderPrice'] == self.limit_up) &
                    (self.order['orderKind'] == 'A') &
                    (self.order['functionCode'] == 'B')]['orderOriNo'].to_list()
        filtered_orders = self.order[(self.order['orderOriNo'].isin(order_ids)) & (self.order['time'] <= self.predict_t)].copy()
        filtered_orders['orderAmount'] = np.where(
            filtered_orders['orderKind'] == 'A', 
            filtered_orders['orderPrice'] * filtered_orders['orderVolume'] / 10000,
            np.where(filtered_orders['orderKind'] == 'D', 
                    -filtered_orders['orderPrice'] * filtered_orders['orderVolume'] / 10000, 
                    0)
        )
        total_amount = filtered_orders['orderAmount'].sum()

        filtered_trades = self.trade[(self.trade['bidOrder'].isin(order_ids)) & (self.trade['time'] <= self.predict_t) & (self.trade['bsFlag'] == 'S')].copy()
        filtered_trades['tradeAmount'] = filtered_trades['tradePrice'] * filtered_trades['tradeVolume'] / 10000
        traded_amount = filtered_trades['tradeAmount'].sum()
        
        return (total_amount - traded_amount) / total_amount if total_amount > 0 else 0
    
    # predict_t距离封板时刻的时间(毫秒)
    def factor005(self):
        return time_sub(self.predict_t, self.lock_time)
    
    # 封板后到predict_t时刻之间的涨停价的挂单数
    def factor006(self):
        order = self.order[(self.order['time'] >= self.lock_time) & 
                           (self.order['time'] <= self.predict_t) &
                           (self.order['orderPrice'] == self.limit_up) &
                           (self.order['orderKind'] == 'A')].copy()
        return len(order)
    
    # 封板后到predict_t时刻之间的涨停价的撤单数
    def factor007(self):
        order = self.order[(self.order['time'] >= self.lock_time) & 
                           (self.order['time'] <= self.predict_t) &
                           (self.order['orderPrice'] == self.limit_up) &
                           (self.order['orderKind'] == 'D')].copy()
        return len(order)

    # 封板后到predict_t时刻之间的涨停价单笔最大撤单量
    def factor008(self):
        order = self.order[(self.order['time'] >= self.lock_time) & 
                           (self.order['time'] <= self.predict_t) &
                           (self.order['orderPrice'] == self.limit_up) &
                           (self.order['orderKind'] == 'D')].copy()
        return order['orderVolume'].max()
    
    # 封板后到predict_t时刻之间的涨停价单笔最大成交量
    def factor009(self):
        trade = self.trade[(self.trade['time'] >= self.lock_time) & 
                           (self.trade['time'] <= self.predict_t) &
                           (self.trade['tradePrice'] == self.limit_up) &
                           (self.trade['bsFlag'] == 'S')].copy()
        return trade['tradeVolume'].max()

    # 封板后到predict_t时刻之间逐笔委托的更新频率
    def factor010(self):
        order = self.order[(self.order['time'] >= self.lock_time) & 
                           (self.order['time'] <= self.predict_t)]
        return time_sub(order['time'].iloc[-1], order['time'].iloc[0]) / len(order)

    # 封板后到predict_t时刻之间逐笔成交的更新频率
    def factor011(self):
        trade = self.trade[(self.trade['time'] >= self.lock_time) & 
                           (self.trade['time'] <= self.predict_t)]
        return time_sub(trade['time'].iloc[-1], trade['time'].iloc[0]) / len(trade)

    # 在predict_t时刻时，自己单子的位置(用0.8分位数来进行过滤)
    def factor012(self):
        t_id = self.order[self.order['time'] > self.t].iloc[0]['orderOriNo']
        temp_order = self.order[(self.order['time'] >= self.lock_time) &
                    (self.order['time'] <= self.predict_t) &
                    (self.order['orderPrice'] == self.limit_up) &
                    (self.order['orderKind'] == 'A')].copy()
        v_threshold = temp_order['orderVolume'].quantile(0.8)
        temp_order = temp_order[temp_order['orderVolume'] > v_threshold]
        temp_trade = self.trade[(self.trade['time'] >= self.lock_time) &
                    (self.trade['time'] <= self.predict_t) &
                    (self.trade['tradePrice'] == self.limit_up) &
                    (self.trade['bsFlag'] == 'S')].copy()
        t_idx = self.order[self.order['orderOriNo'] == t_id].index[0]
        if not t_idx in temp_order.index:
            temp_order = pd.concat([temp_order, self.order[self.order['orderOriNo'] == t_id]]).sort_index()
        last_id = temp_trade.iloc[-1]['bidOrder']
        result = np.searchsorted(temp_order['orderOriNo'], t_id, side='right') - np.searchsorted(temp_order['orderOriNo'], last_id, side='right')
        return result
    
    # 封板后到predict_t时刻之间非涨停价的撤单金额
    def factor013(self):
        temp_order = self.order[(self.order['time'] >= self.lock_time) & 
                            (self.order['time'] <= self.predict_t) &
                            (self.order['orderKind'] == 'D') &
                            (self.order['orderPrice'] != self.limit_up)].copy()
        result = (temp_order['orderPrice'] * temp_order['orderVolume']).sum() / 10000
        return result
    
    # 封板后到predict_t时刻之间非涨停价的撤单数
    def factor014(self):
        temp_order = self.order[(self.order['time'] >= self.lock_time) & 
                            (self.order['time'] <= self.predict_t) &
                            (self.order['orderKind'] == 'D') &
                            (self.order['orderPrice'] != self.limit_up)].copy()
        return len(temp_order)
    
    # 封板后到predict_t时刻之间非涨停价的挂单金额
    def factor015(self):
        temp_order = self.order[(self.order['time'] >= self.lock_time) & 
                            (self.order['time'] <= self.predict_t) &
                            (self.order['orderKind'] == 'A') &
                            (self.order['orderPrice'] != self.limit_up)].copy()
        result = (temp_order['orderPrice'] * temp_order['orderVolume']).sum() / 10000
        return result
    
    # 封板后到predict_t时刻之间非涨停价的挂单数
    def factor016(self):
        temp_order = self.order[(self.order['time'] >= self.lock_time) & 
                            (self.order['time'] <= self.predict_t) &
                            (self.order['orderKind'] == 'A') &
                            (self.order['orderPrice'] != self.limit_up)].copy()
        return len(temp_order)
    
    # 封板后到predict_t时刻之间涨停价订单平均每笔成交量
    def factor017(self):
        temp_trade = self.trade[(self.trade['time'] >= self.lock_time) &
                            (self.trade['time'] <= self.predict_t) &
                            (self.trade['tradePrice'] == self.limit_up)].copy()
        return temp_trade['tradeVolume'].mean()

    # 封板后到predict_t时刻之间涨停价订单平均每笔成交金额
    def factor018(self):
        temp_trade = self.trade[(self.trade['time'] >= self.lock_time) &
                            (self.trade['time'] <= self.predict_t) &
                            (self.trade['tradePrice'] == self.limit_up)].copy()
        return (temp_trade['tradeVolume'] * temp_trade['tradePrice']).mean() / 10000
    
    # 封板后到predict_t时刻之间涨停价订单的成交笔数
    def factor019(self):
        temp_trade = self.trade[(self.trade['time'] >= self.lock_time) &
                            (self.trade['time'] <= self.predict_t) &
                            (self.trade['tradePrice'] == self.limit_up)].copy()
        return len(temp_trade)
    
    # 封板后到predict_t时刻之间非涨停价订单的成交笔数
    def factor020(self):
        temp_trade = self.trade[(self.trade['time'] >= self.lock_time) &
                            (self.trade['time'] <= self.predict_t) &
                            (self.trade['tradePrice'] != self.limit_up)].copy()
        return len(temp_trade)
    
    # 封板订单的委托金额
    def factor021(self):
        temp_order = self.order[self.order['bizIndex'] == self.lock_id]
        return temp_order['orderVolume'].item() * temp_order['orderPrice'].item() / 10000

    # 封板订单的委托金额
    def factor022(self):
        temp_order = self.order[self.order['bizIndex'] == self.lock_id]
        return temp_order['orderVolume'].item()

    # 封板订单成交时间差
    def factor023(self):
        order_id = self.order[self.order['bizIndex'] == self.lock_id]['orderOriNo'].item()
        temp_body = self.trade[self.trade['bidOrder'] == order_id]
        return time_sub(temp_body['time'].iloc[-1], temp_body['time'].iloc[0])
    
    # 封板后到predict_t时刻之间涨停价订单成交额的0.1分位数
    def factor024(self):
        temp_trade = self.trade[(self.trade['time'] >= self.lock_time) &
                    (self.trade['time'] <= self.predict_t) &
                    (self.trade['tradePrice'] == self.limit_up)].copy()
        return temp_trade['tradeVolume'].quantile(0.1) * self.limit_up / 10000
    
    # 封板后到predict_t时刻之间涨停价订单成交额的0.5分位数
    def factor025(self):
        temp_trade = self.trade[(self.trade['time'] >= self.lock_time) &
                    (self.trade['time'] <= self.predict_t) &
                    (self.trade['tradePrice'] == self.limit_up)].copy()
        return temp_trade['tradeVolume'].quantile(0.5) * self.limit_up / 10000
    
    # 封板后到predict_t时刻之间涨停价订单成交额的0.9分位数
    def factor026(self):
        temp_trade = self.trade[(self.trade['time'] >= self.lock_time) &
                    (self.trade['time'] <= self.predict_t) &
                    (self.trade['tradePrice'] == self.limit_up)].copy()
        return temp_trade['tradeVolume'].quantile(0.9) * self.limit_up / 10000
    
    # 封板后到predict_t时刻之间涨停价订单成交额的0.95分位数
    def factor027(self):
        temp_trade = self.trade[(self.trade['time'] >= self.lock_time) &
                    (self.trade['time'] <= self.predict_t) &
                    (self.trade['tradePrice'] == self.limit_up)].copy()
        return temp_trade['tradeVolume'].quantile(0.95) * self.limit_up / 10000
    
    # 封板后到predict_t时刻之间涨停价订单发单到撤单时间小于10ms的总撤单量
    def factor028(self):
        vol_sum = 0
        cancel_order_ids = self.order[(self.order['time'] >= self.lock_time) &
                                (self.order['time'] <= self.predict_t) &
                                (self.order['orderPrice'] == self.limit_up) &
                                (self.order['orderKind'] == 'D')]['orderOriNo'].to_list()
        for cancel_order_id in cancel_order_ids:
            temp_order = self.order[self.order['orderOriNo'] == cancel_order_id]
            time_diff = time_sub(temp_order['time'].iloc[-1], temp_order['time'].iloc[0])
            if time_diff == 0:
                vol_sum += temp_order['orderVolume'].iloc[0]
        return vol_sum
    
    # 封板后到predict_t时刻之间涨停价订单发单到撤单时间小于10ms的总撤单金额
    def factor029(self):
        vol_amount = 0
        cancel_order_ids = self.order[(self.order['time'] >= self.lock_time) &
                                (self.order['time'] <= self.predict_t) &
                                (self.order['orderPrice'] == self.limit_up) &
                                (self.order['orderKind'] == 'D')]['orderOriNo'].to_list()
        for cancel_order_id in cancel_order_ids:
            temp_order = self.order[self.order['orderOriNo'] == cancel_order_id]
            time_diff = time_sub(temp_order['time'].iloc[-1], temp_order['time'].iloc[0])
            if time_diff == 0:
                vol_amount += temp_order['orderVolume'].iloc[0] * self.limit_up / 10000
        return vol_amount
    
    # 封板后到predict_t时刻之间涨停价订单发单到撤单时间小于10ms的总撤单数
    def factor030(self):
        vol_count = 0
        cancel_order_ids = self.order[(self.order['time'] >= self.lock_time) &
                                (self.order['time'] <= self.predict_t) &
                                (self.order['orderPrice'] == self.limit_up) &
                                (self.order['orderKind'] == 'D')]['orderOriNo'].to_list()
        for cancel_order_id in cancel_order_ids:
            temp_order = self.order[self.order['orderOriNo'] == cancel_order_id]
            time_diff = time_sub(temp_order['time'].iloc[-1], temp_order['time'].iloc[0])
            if time_diff == 0:
                vol_count += 1
        return vol_count
    
    # 封板后到predict_t时刻之间涨停价订单加权平均撤单时间
    def factor031(self):
        total_weighted_time = 0
        total_volume = 0
        cancel_order_ids = self.order[(self.order['time'] >= self.lock_time) &
                        (self.order['time'] <= self.predict_t) &
                        (self.order['orderPrice'] == self.limit_up) &
                        (self.order['orderKind'] == 'D')]['orderOriNo'].to_list()
        for cancel_order_id in cancel_order_ids:
            temp_order = self.order[self.order['orderOriNo'] == cancel_order_id]
            duration = time_sub(temp_order['time'].iloc[-1], temp_order['time'].iloc[0])
            volume = temp_order['orderVolume'].iloc[0]
            total_weighted_time += duration * volume
            total_volume += volume
        result = total_weighted_time / total_volume if total_volume > 0 else 0
        return result
    
    # 封板时刻指数增长模型的alpha
    def factor032(self):
        s = self.cross_section[(self.cross_section['time'] == self.lock_time) &
                                (self.cross_section['委买总额'] > 0)]['委买总额']
        if s.empty:
            return np.nan
        Q0 = s.iloc[0]
        Qt = s.iloc[-1]
        t = len(s)
        alpha = np.log(Qt / Q0) / t
        return alpha
    
    # predict_t时刻指数增长模型的alpha
    def factor033(self):
        s = self.cross_section[(self.cross_section['time'] == self.predict_t) &
                                (self.cross_section['委买总额'] > 0)]['委买总额']
        if s.empty:
            return np.nan
        Q0 = s.iloc[0]
        Qt = s.iloc[-1]
        t = len(s)
        alpha = np.log(Qt / Q0) / t
        return alpha
    
    # 封板后到predict_t时刻之间截面数据的count
    def factor034(self):
        temp_cross_section = self.cross_section[(self.cross_section['time'] >= self.lock_time) &
                                                (self.cross_section['time'] <= self.predict_t)]
        return len(temp_cross_section)
    
    # 封板后到predict_t时刻之间截面数据的更新频率
    def factor035(self):
        temp_cross_section = self.cross_section[(self.cross_section['time'] >= self.lock_time) &
                                                (self.cross_section['time'] <= self.predict_t)]
        return len(temp_cross_section) / time_sub(self.predict_t, self.lock_time) if time_sub(self.predict_t, self.lock_time) != 0 else len(self.cross_section)
    
    # 封板后到predict_t时刻的截面数据做差分, 计算predict_t时刻的VaR95
    def factor036(self):
        temp_cross_section = self.cross_section[(self.cross_section['time'] >= self.lock_time) &
                                                (self.cross_section['time'] <= self.predict_t)]
        s = (temp_cross_section['委买总额'] / temp_cross_section['委买总额'].shift(1) - 1).fillna(0).copy()
        var_95 = s.quantile(0.05)
        return var_95
    
    # 封板后到predict_t时刻的截面数据做差分, 计算predict_t时刻的cVaR95
    def factor037(self):
        temp_cross_section = self.cross_section[(self.cross_section['time'] >= self.lock_time) &
                                                (self.cross_section['time'] <= self.predict_t)]
        s = (temp_cross_section['委买总额'] / temp_cross_section['委买总额'].shift(1) - 1).fillna(0).copy()
        var_95 = s.quantile(0.05)
        return s[s < var_95].mean()
    
    # 封板后到predict_t时刻之间的上升梯度
    def factor038(self):
        temp_cross_section = self.cross_section[(self.cross_section['time'] >= self.lock_time) &
                                                (self.cross_section['time'] <= self.predict_t)]
        max_amount_time = temp_cross_section.loc[temp_cross_section['委买总额'].idxmax(), 'time']
        max_amount = temp_cross_section.loc[temp_cross_section['委买总额'].idxmax(), '委买总额']
        return max_amount / time_sub(max_amount_time, self.lock_time) if time_sub(max_amount_time, self.lock_time) != 0 else max_amount
    
    # 封板后到predict_t时刻之间的下降梯度
    def factor039(self):
        temp_cross_section = self.cross_section[(self.cross_section['time'] >= self.lock_time) &
                                                (self.cross_section['time'] <= self.predict_t)]
        max_amount_time = temp_cross_section.loc[temp_cross_section['委买总额'].idxmax(), 'time']
        max_amount = temp_cross_section.loc[temp_cross_section['委买总额'].idxmax(), '委买总额']
        last_amount = temp_cross_section['委买总额'].iloc[-1]
        return (last_amount - max_amount) / time_sub(self.predict_t, max_amount_time) if time_sub(self.predict_t, max_amount_time) != 0 else (last_amount - max_amount)
    
    # 封板后到predict_t时刻之间委买总额的偏度
    def factor040(self):
        temp_cross_section = self.cross_section[(self.cross_section['time'] >= self.lock_time) &
                                                (self.cross_section['time'] <= self.predict_t)]
        return temp_cross_section['委买总额'].skew()
    
    # 封板后70ms内涨停价新增挂单量除以封板后70ms内涨停价新增挂单数
    def factor041(self):
        if time_sub(self.predict_t, self.lock_time) < 70:
            return np.nan
        temp_order = self.order[(self.order['time'] >= self.lock_time) &
                                (self.order['time'] <= time_add(self.lock_time, 70)) &
                                (self.order['orderKind'] == 'A') &
                                (self.order['orderPrice'] == self.limit_up) &
                                (self.order['functionCode'] == 'B')].copy()
        return temp_order['orderVolume'].sum() / len(temp_order)
    
    # 封板后100ms时刻的上升梯度
    def factor042(self):
        if time_sub(self.predict_t, self.lock_time) < 100:
            return np.nan
        temp_cross_section = self.cross_section[(self.cross_section['time'] >= self.lock_time) &
                                                (self.cross_section['time'] <= time_add(self.lock_time, 100))].copy()
        denominator = time_sub(temp_cross_section['time'].iloc[-1], temp_cross_section['time'].iloc[0])
        peak = temp_cross_section['委买总额'].max()
        numerator = peak - temp_cross_section['委买总额'].iloc[0]
        return numerator / denominator if denominator != 0 else numerator
    
    # 封板后到predict_t时刻的主卖金额
    def factor043(self):
        temp_trade = self.trade[(self.trade['time'] >= self.lock_time) &
                                (self.trade['time'] <= self.predict_t) &
                                (self.trade['bsFlag'] == 'S') &
                                (self.trade['tradePrice'] == self.limit_up)].copy()
        return temp_trade['tradeVolume'].sum() * self.limit_up / 10000
    

    # 封板后到predict_t时刻之间涨停价主卖订单bidOrder的数量
    def factor044(self):
        temp_trade = self.trade[(self.trade['time'] >= self.lock_time) &
                            (self.trade['time'] <= self.predict_t) &
                            (self.trade['tradePrice'] == self.limit_up) &
                            (self.trade['bsFlag'] == 'S')].copy()
        return len(temp_trade['bidOrder'].unique())

    # 封板后到predict_t时刻之间涨停价主卖订单askOrder的数量
    def factor045(self):
        temp_trade = self.trade[(self.trade['time'] >= self.lock_time) &
                            (self.trade['time'] <= self.predict_t) &
                            (self.trade['tradePrice'] == self.limit_up) &
                            (self.trade['bsFlag'] == 'S')].copy()
        return len(temp_trade['askOrder'].unique())

    # 封板后到predict_t时刻之间涨停价主卖订单bidOrder的数量除以askOrder的数量
    def factor046(self):
        temp_trade = self.trade[(self.trade['time'] >= self.lock_time) &
                            (self.trade['time'] <= self.predict_t) &
                            (self.trade['tradePrice'] == self.limit_up) &
                            (self.trade['bsFlag'] == 'S')].copy()
        return len(temp_trade['bidOrder'].unique()) / len(temp_trade['askOrder'].unique()) if len(temp_trade['askOrder'].unique()) != 0 else np.nan
    
    # 封板后到predict_t时刻第一次出现涨停价订单撤单时订单的持续时间
    def factor047(self):
        temp_order = self.order[(self.order['time'] >= self.lock_time) &
                                (self.order['time'] <= self.predict_t) &
                                (self.order['orderPrice'] == self.limit_up)].copy()
        order_id = temp_order[temp_order['orderKind'] == 'D']['orderOriNo'].iloc[0]
        temp_order = temp_order[temp_order['orderOriNo'] == order_id]
        if temp_order.empty:
            return np.nan
        return time_sub(temp_order['time'].iloc[-1], temp_order['time'].iloc[0])
    
    # 封板后到predict_t时刻第一次出现涨停价订单撤单时距离封板时间差
    def factor048(self):
        temp_order = self.order[(self.order['time'] >= self.lock_time) &
                                (self.order['time'] <= self.predict_t) &
                                (self.order['orderPrice'] == self.limit_up)].copy()
        order_id = temp_order[temp_order['orderKind'] == 'D']['orderOriNo'].iloc[0]
        temp_order = temp_order[temp_order['orderOriNo'] == order_id]
        if temp_order.empty:
            return np.nan
        return time_sub(temp_order['time'].iloc[-1], self.lock_time)
    
    # 封板后到predict_t时刻第一次出现大额涨停价订单(0.8分位数判定)撤单时订单的持续时间
    def factor049(self):
        temp_order = self.order[(self.order['time'] >= self.lock_time) &
                                (self.order['time'] <= self.predict_t) &
                                (self.order['orderPrice'] == self.limit_up)].copy()
        threshold = temp_order[temp_order['orderKind'] == 'A']['orderVolume'].quantile(0.8)
        order_id = temp_order[(temp_order['orderVolume'] >= threshold) & (temp_order['orderKind'] == 'D')]['orderOriNo'].iloc[0]
        temp_order = self.order[self.order['orderOriNo'] == order_id]
        if temp_order.empty:
            return np.nan
        return time_sub(temp_order['time'].iloc[-1], temp_order['time'].iloc[0])

    # 封板后到predict_t时刻第一次出现大额涨停价订单(0.8分位数判定)撤单时距离封板时间差
    def factor050(self):
        temp_order = self.order[(self.order['time'] >= self.lock_time) &
                                (self.order['time'] <= self.predict_t) &
                                (self.order['orderPrice'] == self.limit_up)].copy()
        threshold = temp_order[temp_order['orderKind'] == 'A']['orderVolume'].quantile(0.8)
        order_id = temp_order[(temp_order['orderVolume'] >= threshold) & (temp_order['orderKind'] == 'D')]['orderOriNo'].iloc[0]
        temp_order = self.order[self.order['orderOriNo'] == order_id]
        if temp_order.empty:
            return np.nan
        return time_sub(temp_order['time'].iloc[-1], self.lock_time)
    
    # 9:30到封板时刻之间trade的vwap的mean
    def factor053(self):
        temp_trade = self.trade[(self.trade['time'] >= 93000000) &
                                (self.trade['time'] <= self.lock_time)].copy()
        vwap_ratio = (temp_trade['tradePrice'] * temp_trade['tradeVolume']).cumsum() / temp_trade['tradeVolume'].cumsum() / self.limit_up
        return vwap_ratio.mean()

    # 9:30到封板时刻之间trade的vwap的std
    def factor054(self):
        temp_trade = self.trade[(self.trade['time'] >= 93000000) &
                                (self.trade['time'] <= self.lock_time)].copy()
        vwap_ratio = (temp_trade['tradePrice'] * temp_trade['tradeVolume']).cumsum() / temp_trade['tradeVolume'].cumsum() / self.limit_up
        return vwap_ratio.std()
    
    # 9:30到封板时刻之间trade的vwap的skew
    def factor055(self):
        temp_trade = self.trade[(self.trade['time'] >= 93000000) &
                                (self.trade['time'] <= self.lock_time)].copy()
        vwap_ratio = (temp_trade['tradePrice'] * temp_trade['tradeVolume']).cumsum() / temp_trade['tradeVolume'].cumsum() / self.limit_up
        return vwap_ratio.skew()
    
    # 9:30到封板时刻之间trade的vwap的kurt
    def factor056(self):
        temp_trade = self.trade[(self.trade['time'] >= 93000000) &
                                (self.trade['time'] <= self.lock_time)].copy()
        vwap_ratio = (temp_trade['tradePrice'] * temp_trade['tradeVolume']).cumsum() / temp_trade['tradeVolume'].cumsum() / self.limit_up
        return vwap_ratio.kurt()
    
    # 9:30到封板时刻之间trade的vwap的kurt
    def factor057(self):
        temp_trade = self.trade[(self.trade['time'] >= 93000000) &
                                (self.trade['time'] <= self.lock_time)].copy()
        vwap_ratio = (temp_trade['tradePrice'] * temp_trade['tradeVolume']).cumsum() / temp_trade['tradeVolume'].cumsum() / self.limit_up
        return vwap_ratio.quantile(0.25)
    
    # 9:30到封板时刻之间trade的vwap的kurt
    def factor058(self):
        temp_trade = self.trade[(self.trade['time'] >= 93000000) &
                                (self.trade['time'] <= self.lock_time)].copy()
        vwap_ratio = (temp_trade['tradePrice'] * temp_trade['tradeVolume']).cumsum() / temp_trade['tradeVolume'].cumsum() / self.limit_up
        return vwap_ratio.median()
    
    # 9:30到封板时刻之间trade的vwap的kurt
    def factor059(self):
        temp_trade = self.trade[(self.trade['time'] >= 93000000) &
                                (self.trade['time'] <= self.lock_time)].copy()
        vwap_ratio = (temp_trade['tradePrice'] * temp_trade['tradeVolume']).cumsum() / temp_trade['tradeVolume'].cumsum() / self.limit_up
        return vwap_ratio.quantile(0.75)
    
    # 9:30到封板时刻之间trade的vwap的min
    def factor060(self):
        temp_trade = self.trade[(self.trade['time'] >= 93000000) &
                                (self.trade['time'] <= self.lock_time)].copy()
        vwap_ratio = (temp_trade['tradePrice'] * temp_trade['tradeVolume']).cumsum() / temp_trade['tradeVolume'].cumsum() / self.limit_up
        return vwap_ratio.min()
    
    # 9:30到封板时刻之间trade的vwap的max
    def factor061(self):
        temp_trade = self.trade[(self.trade['time'] >= 93000000) &
                                (self.trade['time'] <= self.lock_time)].copy()
        vwap_ratio = (temp_trade['tradePrice'] * temp_trade['tradeVolume']).cumsum() / temp_trade['tradeVolume'].cumsum() / self.limit_up
        return vwap_ratio.max()
    
